# [TEST] TEST RUN - Assignment 2: PEFT for LLMs

**Purpose:** Quick validation (10-15 minutes) to verify all experiments can run

**What this tests:**
- [OK] Environment setup
- [OK] AdapterP bug fix
- [OK] All 4 model configurations
- [OK] Model validation
- [OK] Results collection

**After this succeeds, run the FULL notebook for complete results.**

In [ ]:
import os
import sys
import json
import time
import subprocess
import re
import pandas as pd
from datetime import datetime

print("="*80)
print("[TEST] TEST RUN - Assignment 2: Parameter-Efficient Fine-Tuning")
print("="*80)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()

# =============================================================================
# STEP 1: Check GPU
# =============================================================================
print("[STEP 1/8] Checking GPU availability...")
try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        print(f"[+] GPU: {gpu_name}")
        print(f"  CUDA version: {torch.version.cuda}")
        print(f"  PyTorch version: {torch.__version__}")
    else:
        print("[WARN] No GPU detected. Training will be very slow.")
except Exception as e:
    print(f"[WARN] Error checking GPU: {e}")
print()

# =============================================================================
# STEP 2: Setup Environment
# =============================================================================
print("[STEP 2/8] Setting up environment...")
os.chdir('/kaggle/working')
work_dir = os.path.join(os.getcwd(), 'AI6130_Assignment2')
print(f"[+] Working directory: {work_dir}")
print()

# =============================================================================
# STEP 3: Clone Repository
# =============================================================================
print("[STEP 3/8] Cloning repository...")
if os.path.exists(work_dir):
    print("[+] Repository already exists, skipping clone")
else:
    result = subprocess.run(
        ['git', 'clone', 'https://github.com/duyngtr16061999/AI6130_Assignment2.git'],
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        print("[+] Repository cloned")
    else:
        print(f"[WARN] Clone warning: {result.stderr[:200]}")
        print("  (This may be OK if repo already exists)")
os.chdir(work_dir)
print()

# =============================================================================
# STEP 4: Install Dependencies
# =============================================================================
print("[STEP 4/8] Installing dependencies...")
print("  Note: Warnings about dependency conflicts are OK")
packages = [
    'fire',
    'datasets',
    'sentencepiece',
    'protobuf',
    'transformers==4.38.0',
    'accelerate==0.28.0'
]

for pkg in packages:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', pkg],
        check=False  # Don't fail on warnings
    )

# Reinstall peft from source
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-q', '-y', 'peft'], check=False)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/huggingface/peft.git@main'],
    check=False
)
print("[+] Dependencies installed")
print()

# =============================================================================
# STEP 5: Patch finetune.py for AdapterP Bug
# =============================================================================
print("[STEP 5/8] Patching finetune.py for AdapterP bug...")
finetune_path = 'finetune.py'
with open(finetune_path, 'r') as f:
    content = f.read()

if 'PATCHED_FOR_ADAPTERP' not in content:
    lines = content.split('\n')
    patched_lines = []
    
    for i, line in enumerate(lines):
        # Add patch before 'model = get_peft_model(model, config)'
        if 'model = get_peft_model(model, config)' in line:
            indent = len(line) - len(line.lstrip())
            patched_lines.append(' ' * indent + '# PATCHED_FOR_ADAPTERP: Initialize config if not set')
            patched_lines.append(' ' * indent + 'if "config" not in locals():')
            patched_lines.append(' ' * indent + '    config = None')
            patched_lines.append(' ' * indent + 'if config is None:')
            patched_lines.append(' ' * indent + '    raise ValueError(f"Unknown adapter: {adapter_name}")')
            patched_lines.append('')
        patched_lines.append(line)
    
    with open(finetune_path, 'w') as f:
        f.write('\n'.join(patched_lines))
    print("[+] finetune.py patched for AdapterP")
else:
    print("[+] finetune.py already patched")
print()

# =============================================================================
# STEP 6: Clean Previous Models
# =============================================================================
print("[STEP 6/8] Cleaning previous models...")
models_dir = './trained_models'
if os.path.exists(models_dir):
    import shutil
    shutil.rmtree(models_dir)
    print("[+] Removed old models")
os.makedirs(models_dir, exist_ok=True)
print("[+] Models directory ready")
print()

# =============================================================================
# STEP 7: TEST TRAINING - Minimal Steps
# =============================================================================
print("[STEP 7/8] TEST TRAINING - Minimal runs to verify setup...")
print("  Note: Using only 1 training step per model (just to verify it works)")
print()

# Test experiments: All 4 configurations
TEST_EXPERIMENTS = [
    {
        'name': 'OpenLLaMA-LoRA',
        'base_model': 'openlm-research/open_llama_3b',
        'adapter': 'lora',
        'output_dir': './trained_models/openlm-lora',
        'max_steps': 1  # Just 1 step for test
    },
    {
        'name': 'OpenLLaMA-AdapterP',
        'base_model': 'openlm-research/open_llama_3b',
        'adapter': 'AdapterP',
        'output_dir': './trained_models/openlm-adapterp',
        'max_steps': 1
    },
    {
        'name': 'TinyLlama-LoRA',
        'base_model': 'TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T',
        'adapter': 'lora',
        'output_dir': './trained_models/tinyllama-lora',
        'max_steps': 1
    },
    {
        'name': 'TinyLlama-AdapterP',
        'base_model': 'TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T',
        'adapter': 'AdapterP',
        'output_dir': './trained_models/tinyllama-adapterp',
        'max_steps': 1
    }
]

test_results = {'train': [], 'eval': []}

for i, exp in enumerate(TEST_EXPERIMENTS, 1):
    print("="*80)
    print(f"Testing {i}/{len(TEST_EXPERIMENTS)}: {exp['name']}")
    print("="*80)
    
    cmd = [
        sys.executable,
        'finetune.py',
        '--base_model', exp['base_model'],
        '--data_path', 'math_7k.json',
        '--output_dir', exp['output_dir'],
        '--batch_size', '8',
        '--micro_batch_size', '2',
        '--num_epochs', '1',
        '--learning_rate', '3e-4',
        '--cutoff_len', '256',
        '--val_set_size', '120',
        '--adapter_name', exp['adapter'],
        '--target_modules', '["q_proj", "k_proj", "v_proj", "up_proj", "down_proj"]',
        '--lora_r', '16',
        '--lora_alpha', '16',
        '--lora_dropout', '0.05',
        '--max_steps', str(exp['max_steps'])  # Only 1 step for test
    ]
    
    start_time = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)  # 5 min timeout
    elapsed = time.time() - start_time
    
    if result.returncode == 0:
        print(f"[+] Training completed in {elapsed:.1f}s")
        test_results['train'].append({'model': exp['name'], 'status': 'SUCCESS', 'time': elapsed})
    else:
        print(f"[X] Training FAILED after {elapsed:.1f}s")
        print(f"Error: {result.stderr[-500:]}")
        test_results['train'].append({'model': exp['name'], 'status': 'FAILED', 'time': elapsed})
    print()

# =============================================================================
# STEP 8: TEST EVALUATION - Quick Check
# =============================================================================
print("[STEP 8/8] TEST EVALUATION - Quick check on 1 sample...")
print()

def is_model_valid(model_dir):
    """Check if model has required files"""
    if not os.path.exists(model_dir):
        return False
    required_files = ['adapter_config.json', 'adapter_model.bin']
    for file in required_files:
        file_path = os.path.join(model_dir, file)
        if not os.path.exists(file_path):
            return False
        if os.path.getsize(file_path) == 0:
            return False
    return True

# Test evaluation on one dataset
test_dataset = 'AddSub'

for exp in TEST_EXPERIMENTS:
    print(f"Testing evaluation: {exp['name']} on {test_dataset}")
    
    if not is_model_valid(exp['output_dir']):
        print(f"[X] Model not valid, skipping")
        test_results['eval'].append({'model': exp['name'], 'dataset': test_dataset, 'status': 'SKIPPED'})
        print()
        continue
    
    cmd = [
        sys.executable,
        'evaluate.py',
        '--base_model', exp['base_model'],
        '--lora_weights', exp['output_dir'],
        '--dataset', test_dataset
    ]
    
    start_time = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)  # 1 min timeout
    elapsed = time.time() - start_time
    
    if result.returncode == 0:
        print(f"[+] Evaluation completed in {elapsed:.1f}s")
        test_results['eval'].append({'model': exp['name'], 'dataset': test_dataset, 'status': 'SUCCESS', 'time': elapsed})
    else:
        print(f"[X] Evaluation FAILED after {elapsed:.1f}s")
        test_results['eval'].append({'model': exp['name'], 'dataset': test_dataset, 'status': 'FAILED', 'time': elapsed})
    print()

# =============================================================================
# TEST RESULTS SUMMARY
# =============================================================================
print("="*80)
print("[TEST] TEST RUN RESULTS")
print("="*80)
print()

print("Training Results:")
train_df = pd.DataFrame(test_results['train'])
print(train_df.to_string(index=False))
print()

print("Evaluation Results:")
eval_df = pd.DataFrame(test_results['eval'])
print(eval_df.to_string(index=False))
print()

# Check if all passed
train_success = all(r['status'] == 'SUCCESS' for r in test_results['train'])
eval_success = all(r['status'] in ['SUCCESS', 'SKIPPED'] for r in test_results['eval'])

if train_success and eval_success:
    print("[OK] ALL TESTS PASSED!")
    print("[OK] Environment is ready for full run")
    print()
    print("> Next step: Run Assignment2_FULL_RUN.ipynb for complete results")
else:
    print("[WARN] SOME TESTS FAILED")
    print("[WARN] Please check errors above before running full version")

print()
print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)